# Indice de refracción del aire

Los calculos presentados en el desarrollo de este notebook están basados en el trabajo presentado por el [NIST](https://emtoolbox.nist.gov/Wavelength/Documentation.asp#AppendixA).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
# Agrega el directorio padre (la raíz del repositorio) al path temporal de Python
sys.path.append(os.path.abspath(os.path.join('..')))

import numpy as np

from src.refraction_index import EdlenRefractiveIndex

In [2]:
def saturation_vapor_pressure_over_water(temperature: float) -> float:
    """
    Paramters
    ---------
    temperature: float
        Temperature in °C.
    """
    k1 = 1.16705214528e3
    k2 = -7.24213167032e5
    k3 = -1.70738469401e1
    k4 = 1.20208247025e4
    k5 = -3.23255503223e6
    k6 = 1.49151086135e1
    k7 = -4.82326573616e03
    k8 = 4.05113405421e05
    k9 = -2.38555575678e-1
    k10 = 6.50175348448e2
    T = temperature + 273.15
    omega = T + (k9 / (T - k10))
    A = omega**2 + k1 * omega + k2
    B = k3 * omega**2 + k4 * omega + k5
    C = k6 * omega**2 + k7 * omega + k8
    X = -B + (B**2 - 4 * A * C)**0.5
    P_sv = 1e6 * (2 * C / X)**4
    return P_sv
    
def saturation_vapor_pressure_over_ice(temperature: float) -> float:
    A1 = -13.928169
    A2 = 34.7078238
    T = temperature + 273.15
    Theta = T / 273.16
    Y = A1 * (1 - Theta**(-1.5)) + A2 * (1 - Theta**(-1.25))
    P_sv = 611.657 * np.exp(Y)
    return P_sv

In [3]:
def humidity_Edlen(t: float, RH: float = None, dew_point: bool = False, frost_point: bool = False) -> float:
    if t >= 0: P_sv = saturation_vapor_pressure_over_water(t)
    else: P_sv = saturation_vapor_pressure_over_ice(t)
    
    if dew_point or frost_point: return P_sv
    else: return (RH / 100) * P_sv 

In [4]:
def index_refraction_Edlen(wavelength: float, t: float, p: float, p_v: float) -> float:
    """
    Parameters
    ----------
    wavelength : float
        Wavelength in vacuum (um)
    t : float
        Temperature (°C)
    p : float
        Pressure (Pascal)
    p_v : float
        Vapor pressure (Pascal)
    """
    A = 8342.54
    B = 2406147
    C = 15998
    D = 96095.43
    E = 0.601
    F = 0.00972
    G = 0.003661
    
    S = 1 / wavelength**2
    n_s = 1 + 10**(-8) * (A + B / (130 - S) + C / (38.9 - S))
    X =  (1 + 10**(-8) * (E - F * t) * p) / (1 + G * t)
    n_tp = 1 + p * (n_s - 1) * X / D
    n = n_tp - 1e-10 * ((292.75) / (t + 273.15)) * (3.7345 - 0.0401 * S) * p_v
    return n
    

In [5]:
def wavelength_in_air(wavelength: float, n: float) -> float: return wavelength / n

In [6]:
# initial parameters
vacuum_wavelength = 633 # nm
vacuum_wavelength_um = vacuum_wavelength / 1000 # um
air_temperature = 20 # °C
atmospheric_pressure = 101325 # Pa
air_humidity = 50 # %

In [7]:
p_v = humidity_Edlen(t=air_temperature, RH=air_humidity)
ntp = index_refraction_Edlen(vacuum_wavelength_um, air_temperature, atmospheric_pressure, p_v)
wavelength_air = wavelength_in_air(vacuum_wavelength_um, ntp)

print(f"Refractive index of air: {ntp:.9f}")
print(f"Wavelength in air: {wavelength_air*1e3:.6f} nm")

Refractive index of air: 1.000271374
Wavelength in air: 632.828267 nm


In [8]:
env = EdlenRefractiveIndex(temperature=air_temperature, pressure=atmospheric_pressure, relative_humidity=air_humidity)
index_refraction = env.index_refraction(vacuum_wavelength_um)
wavelength_air = env.wavelength_in_air(vacuum_wavelength_um)

print(f"Refractive index of air: {index_refraction:.9f}")
print(f"Wavelength in air: {wavelength_air*1e3:.6f} nm")

Refractive index of air: 1.000271374
Wavelength in air: 632.828267 nm
